# 좋은 도구 이름과 설명

목록·ID·집계 도구의 설명서를 비교합니다.

## 준비

## 코드를 읽기 전에

세 도구는 목록 조회, 단일 ID 조회, 상태별 집계라는 서로 다른 결과를 만듭니다. `name`은 모델이 호출 요청에 쓸 식별자이고, 함수의 문서 문자열은 `description`이 됩니다. 이 노트북에서는 타입 힌트가 `args`의 필드 이름과 자료형으로 변환됩니다. `Field(description=...)` 같은 명시적 필드 설명은 사용하지 않았으므로 출력에도 필드 설명이 없습니다. 노트북은 이름·설명·입력 스키마를 출력해 모델에게 실제로 전달되는 설명서를 확인합니다.

설명 길이의 문자 수는 스키마가 입력 문맥의 일부라는 사실을 비교하기 위한 보조값입니다. 문자 수와 토큰 수는 같지 않으므로 이 값을 모델 사용량이나 비용으로 바꾸지 않습니다. 좋은 설명은 길이보다 선택 조건과 반환 범위가 겹치지 않는지가 중요합니다.

In [1]:
# 상위 폴더에서 환경 파일과 공개 주문 CSV를 찾습니다.
from pathlib import Path
from dotenv import load_dotenv

def find_up(name: str) -> Path | None:
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        candidate = folder / name
        if candidate.exists():
            return candidate
    return None

load_dotenv(find_up(".env"))
data_path = find_up("data/orders.csv")
assert data_path is not None, "공개 묶음의 챕터 폴더 또는 notebooks 폴더에서 실행합니다."
print("Python 실습 준비 완료")


Python 실습 준비 완료


### 다음 셀을 읽는 방법

다음 셀은 세 도구가 함께 호출할 `fetch_orders()`를 준비합니다. CSV 경로와 선택 상태를 입력받아 사전 목록을 반환하므로, 뒤의 도구들은 파일 읽기 코드를 반복하지 않고 선택 목적만 다르게 정의할 수 있습니다.


In [2]:
# 세 도구가 함께 사용할 조회 함수를 준비합니다.
from pathlib import Path
from typing import Any

import pandas as pd


def fetch_orders(csv_path: str | Path, *, status: str | None = None) -> list[dict[str, Any]]:
    """CSV 주문을 읽고 선택한 상태의 행을 사전 목록으로 반환합니다."""
    orders = pd.read_csv(csv_path, dtype={"order_id": "string", "customer_id": "string"})
    if status is not None:
        orders = orders.loc[orders["status"] == status]
    return orders.to_dict(orient="records")


### 다음 셀을 읽는 방법

`@tool`은 바로 아래 함수를 `StructuredTool`로 감싸며 함수 이름·문서 문자열·타입 힌트를 설명서로 만듭니다. `list_orders_by_status()`는 상태 문자열을 입력받아 여러 주문 사전을 반환하고, 이 셀에서는 도구 정의만 하므로 모델이나 CSV 조회는 아직 실행되지 않습니다.


In [3]:
# 상태별 목록 도구에 선택 조건을 설명합니다.
from langchain.tools import tool
@tool
def list_orders_by_status(status: str) -> list[dict[str, Any]]:
    """주문 상태(paid, pending, cancelled)로 주문 목록을 조회합니다."""
    return fetch_orders(data_path, status=status)


### 다음 셀을 읽는 방법

다음 셀은 정확한 ID로 한 행을 찾는 도구와 상태별 건수·합계를 계산하는 도구를 정의합니다. 둘 다 `@tool`로 감싸지만 입력 이름과 반환 모양이 다릅니다. 함수 본문은 실제 `invoke()`가 호출될 때 실행됩니다.


In [4]:
# 한 건 조회와 금액 집계의 이름·반환 목적을 구분합니다.
@tool
def get_order_by_id(order_id: str) -> dict:
    """정확한 주문 ID로 주문 한 건의 상태와 금액을 조회합니다."""
    rows = fetch_orders(data_path)
    return next((r for r in rows if r["order_id"] == order_id), {"error": "주문을 찾지 못했습니다."})

@tool
def summarize_order_amounts(status: str) -> dict:
    """주문 상태별 건수와 총 주문 금액을 집계합니다."""
    rows = fetch_orders(data_path, status=status)
    return {"status": status, "count": len(rows), "total": sum(r["amount"] for r in rows)}


## 모델이 보는 설명

In [5]:
# 모델에게 전달할 도구 이름·설명·입력 칸을 확인합니다.
tools = [list_orders_by_status, get_order_by_id, summarize_order_amounts]
for item in tools:
    print({"name": item.name, "description": item.description, "args": item.args})
assert len({item.name for item in tools}) == 3


{'name': 'list_orders_by_status', 'description': '주문 상태(paid, pending, cancelled)로 주문 목록을 조회합니다.', 'args': {'status': {'title': 'Status', 'type': 'string'}}}
{'name': 'get_order_by_id', 'description': '정확한 주문 ID로 주문 한 건의 상태와 금액을 조회합니다.', 'args': {'order_id': {'title': 'Order Id', 'type': 'string'}}}
{'name': 'summarize_order_amounts', 'description': '주문 상태별 건수와 총 주문 금액을 집계합니다.', 'args': {'status': {'title': 'Status', 'type': 'string'}}}


## 설명서 크기

In [6]:
# 문자 수를 비교하며 토큰 사용량과 구분합니다.
import json
for item in tools:
    print(item.name, len(item.description), len(json.dumps(item.args, ensure_ascii=False)))


list_orders_by_status 46 49
get_order_by_id 33 53
summarize_order_amounts 26 49


## 실험 결과 해석

선택 정확도는 다음 절에서 실제로 측정합니다.

## 다른 상황에 적용하기

`search`, `find`, `lookup`처럼 비슷한 이름의 도구가 모두 ‘정보를 찾습니다’라고만 쓰면 모델은 경계를 알기 어렵습니다. `search_articles_by_keyword`는 여러 기사 요약을, `get_article_by_id`는 한 기사의 원문을 반환한다고 구분할 수 있습니다. 설명에 구현 세부를 길게 적기보다 언제 선택하고 무엇을 돌려받는지 명시합니다.